# GOOGL Stock Movement Prediction
## Part 1: Data Collection

This notebook collects raw data from three sources covering January 2010
to present:

| Source | Data | Coverage |
|---|---|---|
| Yahoo Finance (yfinance) | Daily OHLCV prices | 2010–present |
| SEC EDGAR XBRL API | Quarterly financial statements | 2012–present |
| FRED API | Macroeconomic indicators | 2010–present |

**Note on news data:** Finnhub and GDELT were evaluated as news sources.
Finnhub free tier covers only the last ~2 years; GDELT's Doc 2.0 API has
a 3-month rolling window. Neither provides the 2010-present coverage needed.
News is therefore excluded from this project and documented as a limitation.

**Output:**
```
data/raw/
├── price_raw.csv
├── fundamental_raw.csv
└── macro_raw.csv
```

**API keys required (free tier):**
- `FRED_API_KEY` → https://fred.stlouisfed.org/docs/api/api_key.html

In [20]:
!pip install yfinance fredapi pandas numpy tqdm requests -q

In [21]:
import os
import time
import warnings
import requests
import numpy as np
import pandas as pd
from datetime import datetime, date, timedelta
from tqdm import tqdm
import yfinance as yf

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 50)
pd.set_option("display.float_format", "{:.4f}".format)

TICKER = "GOOGL"
GOOGL_CIK = "0001652044"
START_DATE = "2010-01-01"
END_DATE = date.today().strftime("%Y-%m-%d")

RAW_DIR = "data/raw"
os.makedirs(RAW_DIR, exist_ok=True)

# Set these before running, or export as environment variables
FRED_API_KEY = os.environ.get("FRED_API_KEY", "299816ca4ddcfa35c9b1f049a29b2108")

EDGAR_HEADERS = {"User-Agent": "GOOGL-Research khadafi6756pkt@gmail.com"}

print(f"Ticker     : {TICKER}")
print(f"Start date : {START_DATE}")
print(f"End date   : {END_DATE}")
print(f"Output dir : {RAW_DIR}/")
print(f"FRED key   : {'set' if FRED_API_KEY else 'NOT SET — macro section will be skipped'}")

Ticker     : GOOGL
Start date : 2010-01-01
End date   : 2026-06-19
Output dir : data/raw/
FRED key   : set


## Price and Volume Data

**Source:** Yahoo Finance via `yfinance`

`auto_adjust=True` applies retroactive adjustments for:
- The 20:1 stock split in July 2022
- Dividend payouts

This ensures `Close` reflects the true investable return across the full
history, rather than showing an artificial price discontinuity at the
split date.

In [22]:
def collect_price_data() -> pd.DataFrame:
  df = yf.download(
      TICKER,
      start=START_DATE,
      end=END_DATE,
      auto_adjust=True,
      progress=False,
    )
  if isinstance(df.columns, pd.MultiIndex):
    df.columns = df.columns.get_level_values(0)

    df.index = pd.to_datetime(df.index)
    df.index.name = "date"

    return df


def validate_price_data(df: pd.DataFrame) -> None:
    daily_return = df["Close"].pct_change().dropna()
    trading_gaps = df.index.to_series().diff().dt.days.dropna()

    print(f"Rows       : {len(df):,}")
    print(f"Date range : {df.index.min().date()} to {df.index.max().date()}")
    print(f"Columns    : {list(df.columns)}")
    print()
    print("Missing values:")
    print(df.isnull().sum().to_string())
    print()
    print(f"Gaps over 4 trading days   : {int((trading_gaps > 4).sum())}")
    print(f"Close <= 0                 : {int((df['Close'] <= 0).sum())}")
    print(f"Volume <= 0                : {int((df['Volume'] <= 0).sum())}")
    print(f"Daily returns > ±20%       : {int((daily_return.abs() > 0.20).sum())}")


print("Collecting price data...")
price_df = collect_price_data()
validate_price_data(price_df)

price_df.to_csv(f"{RAW_DIR}/price_raw.csv")
print(f"\nSaved: {RAW_DIR}/price_raw.csv")

Rows       : 4,140
Date range : 2010-01-04 to 2026-06-18
Columns    : ['Close', 'High', 'Low', 'Open', 'Volume']

Missing values:
Price
Close     0
High      0
Low       0
Open      0
Volume    0

Gaps over 4 trading days   : 1
Close <= 0                 : 0
Volume <= 0                : 0
Daily returns > ±20%       : 0

Saved: data/raw/price_raw.csv


## Fundamental Data

**Source:** SEC EDGAR Company Facts API (XBRL)

All public US companies file structured financial data with the SEC using
XBRL taxonomy tags. This API returns the complete filing history for every
reported concept — no scraping, no third-party aggregators.

**GOOGL CIK:** `0001652044` (Alphabet Inc.)

**Filing logic:**
- Only `10-Q` (quarterly) and `10-K` (annual) filings are kept
- If a filing was amended (`10-Q/A`, `10-K/A`), the most recently filed
  version is used
- All values are as-reported in USD (not adjusted)

**Known gaps:**
- 4 concepts are reported under alternative GAAP tags by Alphabet and have
  fallback mappings defined below
- Coverage starts at 2012-12-31 because Alphabet's early XBRL filings
  (2010-2011) did not tag all line items at this granularity

In [23]:
# Primary concept → column name
# For each concept, a fallback list is tried in order if the primary tag
# is absent in this company's filing history
CONCEPT_MAP = {
    "Revenues"                                          : ("revenue",              ["RevenueFromContractWithCustomerExcludingAssessedTax"]),
    "GrossProfit"                                       : ("gross_profit",         ["GrossProfitLoss"]),
    "OperatingIncomeLoss"                               : ("operating_income",     []),
    "NetIncomeLoss"                                     : ("net_income",           ["ProfitLoss"]),
    "CostOfRevenue"                                     : ("cost_of_revenue",      ["CostOfGoodsAndServicesSold"]),
    "ResearchAndDevelopmentExpense"                     : ("rd_expense",           ["ResearchAndDevelopmentExpenseExcludingAcquiredInProcessCost"]),
    "SellingGeneralAndAdministrativeExpense"            : ("sga_expense",          ["GeneralAndAdministrativeExpense", "SellingAndMarketingExpense"]),
    "IncomeTaxExpenseBenefit"                           : ("income_tax",           []),
    "InterestExpense"                                   : ("interest_expense",     ["InterestAndDebtExpense"]),
    "DepreciationDepletionAndAmortization"              : ("dda",                  ["DepreciationAndAmortization", "Depreciation"]),
    "Assets"                                            : ("total_assets",         []),
    "Liabilities"                                       : ("total_liabilities",    []),
    "StockholdersEquity"                                : ("stockholders_equity",  ["StockholdersEquityIncludingPortionAttributableToNoncontrollingInterest"]),
    "CashAndCashEquivalentsAtCarryingValue"             : ("cash",                 ["CashAndCashEquivalents"]),
    "ShortTermInvestments"                              : ("short_term_investments", ["AvailableForSaleSecuritiesCurrent", "MarketableSecuritiesCurrent"]),
    "AccountsReceivableNetCurrent"                      : ("accounts_receivable",  ["ReceivablesNetCurrent"]),
    "PropertyPlantAndEquipmentNet"                      : ("ppe_net",              []),
    "Goodwill"                                          : ("goodwill",             []),
    "IntangibleAssetsNetExcludingGoodwill"              : ("intangibles",          []),
    "LongTermDebt"                                      : ("long_term_debt",       ["LongTermDebtNoncurrent"]),
    "LiabilitiesCurrent"                                : ("current_liabilities",  []),
    "AssetsCurrent"                                     : ("current_assets",       []),
    "RetainedEarningsAccumulatedDeficit"                : ("retained_earnings",    []),
    "CommonStockSharesOutstanding"                      : ("shares_outstanding",   []),
    "NetCashProvidedByUsedInOperatingActivities"        : ("operating_cash_flow",  []),
    "PaymentsToAcquirePropertyPlantAndEquipment"        : ("capex",                ["CapitalExpendituresIncurredButNotYetPaid"]),
    "NetCashProvidedByUsedInInvestingActivities"        : ("investing_cash_flow",  []),
    "NetCashProvidedByUsedInFinancingActivities"        : ("financing_cash_flow",  []),
    "EarningsPerShareBasic"                             : ("eps_basic",            []),
    "EarningsPerShareDiluted"                           : ("eps_diluted",          []),
}

print(f"Total concepts defined: {len(CONCEPT_MAP)}")

Total concepts defined: 30


In [24]:
def fetch_company_facts(cik: str) -> dict:
    url = f"https://data.sec.gov/api/xbrl/companyfacts/CIK{cik}.json"
    response = requests.get(url, headers=EDGAR_HEADERS, timeout=60)
    response.raise_for_status()
    return response.json()


def parse_concept(gaap_facts: dict, concept: str, fallbacks: list) -> list | None:
    """
    Try primary concept first, then each fallback in order.
    Returns the raw records list, or None if nothing found.
    """
    for tag in [concept] + fallbacks:
        if tag in gaap_facts:
            units = gaap_facts[tag].get("units", {})
            for unit_key in ["USD", "shares", "USD/shares", "pure"]:
                if unit_key in units:
                    return units[unit_key]
    return None


def records_to_series(records: list, column_name: str) -> pd.Series:
    df = pd.DataFrame(records)
    if "end" not in df.columns or "val" not in df.columns:
        return pd.Series(dtype=float, name=column_name)

    df = df[["end", "val", "form", "filed"]].copy()
    df.columns = ["end_date", "value", "form", "filed"]
    df["end_date"] = pd.to_datetime(df["end_date"])
    df["filed"] = pd.to_datetime(df["filed"])

    df = df[df["form"].isin(["10-Q", "10-K", "10-K/A", "10-Q/A"])]
    df = df[df["end_date"] >= START_DATE]

    # If amended, keep the latest filed version for each reporting period
    df = df.sort_values("filed").drop_duplicates("end_date", keep="last")

    return df.set_index("end_date")["value"].rename(column_name).sort_index()


def collect_fundamental_data() -> pd.DataFrame:
    print("Fetching SEC EDGAR company facts...")
    company_facts = fetch_company_facts(GOOGL_CIK)
    gaap_facts = company_facts["facts"].get("us-gaap", {})
    print(f"Total GAAP concepts available in EDGAR: {len(gaap_facts)}")
    print()

    series_list = []
    found, not_found = [], []

    for concept, (column_name, fallbacks) in CONCEPT_MAP.items():
        records = parse_concept(gaap_facts, concept, fallbacks)
        if records is None:
            not_found.append((concept, column_name))
            continue
        series = records_to_series(records, column_name)
        if not series.empty:
            series_list.append(series)
            found.append(column_name)

    if not_found:
        print(f"Concepts not resolved ({len(not_found)}):")
        for concept, col in not_found:
            print(f"  {concept} -> {col}")
        print()

    df = pd.concat(series_list, axis=1)
    df.index.name = "period_end"
    return df


fundamental_df = collect_fundamental_data()

print(f"Shape      : {fundamental_df.shape}")
print(f"Date range : {fundamental_df.index.min().date()} to {fundamental_df.index.max().date()}")
print()
mv_pct = (fundamental_df.isnull().sum() / len(fundamental_df) * 100).round(1)
print("Missing values per column (%):")
print(mv_pct[mv_pct > 0].to_string() if mv_pct.any() else "  none")
print()
print("Last 3 periods:")
print(fundamental_df.tail(3).to_string())

fundamental_df.to_csv(f"{RAW_DIR}/fundamental_raw.csv")
print(f"\nSaved: {RAW_DIR}/fundamental_raw.csv")

Fetching SEC EDGAR company facts...
Total GAAP concepts available in EDGAR: 523

Concepts not resolved (1):
  GrossProfit -> gross_profit

Shape      : (49, 29)
Date range : 2012-12-31 to 2026-03-31

Missing values per column (%):
revenue                  26.5000
operating_income          2.0000
net_income                2.0000
cost_of_revenue           2.0000
rd_expense                2.0000
sga_expense               2.0000
income_tax                2.0000
interest_expense         18.4000
dda                      69.4000
total_assets             10.2000
total_liabilities        14.3000
stockholders_equity       6.1000
short_term_investments   75.5000
accounts_receivable      10.2000
ppe_net                  42.9000
goodwill                  8.2000
intangibles              28.6000
long_term_debt           77.6000
current_liabilities      10.2000
current_assets           10.2000
retained_earnings        10.2000
shares_outstanding       12.2000
operating_cash_flow       2.0000
capex     

## Macroeconomic Data

**Source:** FRED API (Federal Reserve Economic Data)

These indicators provide macroeconomic context that affects the entire
equity market and technology sector specifically. The series span three
different native frequencies:

| Series | Description | Frequency |
|---|---|---|
| FEDFUNDS | Federal Funds Rate | Monthly |
| DGS10 | 10-Year Treasury Yield | Daily |
| DGS2 | 2-Year Treasury Yield | Daily |
| VIXCLS | CBOE VIX Index | Daily |
| UNRATE | Unemployment Rate | Monthly |
| CPIAUCSL | CPI (Inflation) | Monthly |
| GDPC1 | Real GDP | Quarterly |
| DCOILWTICO | WTI Crude Oil Price | Daily |
| DTWEXBGS | USD Trade-Weighted Index | Daily |
| T10YIE | 10-Year Breakeven Inflation | Daily |
| NASDAQCOM | NASDAQ Composite Index | Daily |
| SP500 | S&P 500 Index | Daily |

Missing values in this raw output are expected — they reflect the native
frequency differences. Forward-filling to daily resolution is handled in
Part 2 (Feature Engineering).

In [25]:
MACRO_SERIES = {
    "FEDFUNDS"   : "fed_funds_rate",
    "DGS10"      : "treasury_10y",
    "DGS2"       : "treasury_2y",
    "VIXCLS"     : "vix",
    "UNRATE"     : "unemployment_rate",
    "CPIAUCSL"   : "cpi",
    "GDPC1"      : "real_gdp",
    "DCOILWTICO" : "oil_price_wti",
    "DTWEXBGS"   : "usd_index",
    "T10YIE"     : "breakeven_inflation_10y",
    "NASDAQCOM"  : "nasdaq_composite",
    "SP500"      : "sp500",
}


def collect_macro_data(api_key: str) -> pd.DataFrame:
    from fredapi import Fred

    fred = Fred(api_key=api_key)
    series_list = []

    for series_id, column_name in tqdm(MACRO_SERIES.items(), desc="FRED series"):
        try:
            s = fred.get_series(series_id, observation_start=START_DATE)
            s.name = column_name
            series_list.append(s)
        except Exception as error:
            print(f"  Warning: {series_id} failed — {error}")

    df = pd.concat(series_list, axis=1)
    df.index = pd.to_datetime(df.index)
    df.index.name = "date"
    df = df[df.index >= START_DATE]

    return df


if not FRED_API_KEY:
    print("FRED_API_KEY not set — skipping macro collection.")
    print("Set it with: FRED_API_KEY = 'your_key_here'")
else:
    macro_df = collect_macro_data(FRED_API_KEY)

    print(f"\nShape      : {macro_df.shape}")
    print(f"Date range : {macro_df.index.min().date()} to {macro_df.index.max().date()}")
    print()

    # Missing values here are expected due to mixed frequencies
    mv_pct = (macro_df.isnull().sum() / len(macro_df) * 100).round(1)
    print("Missing values per column (% — expected due to mixed frequency):")
    print(mv_pct.to_string())

    macro_df.to_csv(f"{RAW_DIR}/macro_raw.csv")
    print(f"\nSaved: {RAW_DIR}/macro_raw.csv")

FRED series: 100%|██████████| 12/12 [00:11<00:00,  1.01it/s]


Shape      : (4351, 12)
Date range : 2010-01-01 to 2026-06-18

Missing values per column (% — expected due to mixed frequency):
fed_funds_rate            95.5000
treasury_10y               5.4000
treasury_2y                5.4000
vix                        4.2000
unemployment_rate         95.5000
cpi                       95.5000
real_gdp                  98.5000
oil_price_wti              5.2000
usd_index                  5.9000
breakeven_inflation_10y    5.4000
nasdaq_composite           4.8000
sp500                     42.2000

Saved: data/raw/macro_raw.csv


## Summary

In [26]:
print("  Data Collection Summary")

files = {
    "price_raw.csv"       : "Daily OHLCV, 2010-present",
    "fundamental_raw.csv" : "Quarterly financials, 2012-present",
    "macro_raw.csv"       : "Macro indicators, 2010-present",
}

n_ok = 0
for fname, description in files.items():
    fpath = f"{RAW_DIR}/{fname}"
    if os.path.exists(fpath):
        df = pd.read_csv(fpath, index_col=0)
        print(f"\n  {fname}")
        print(f"  {description}")
        print(f"  Shape: {df.shape}")
        n_ok += 1
    else:
        print(f"\n  MISSING: {fname}")

print(f"\n  {n_ok}/3 datasets collected")

if n_ok == 3:
    print("\n  Ready for Part 2: Feature Engineering")
elif n_ok >= 2:
    print("\n  Proceeding to Part 2 is possible — check missing dataset above")
else:
    print("\n  Resolve missing datasets before continuing")

  Data Collection Summary

  price_raw.csv
  Daily OHLCV, 2010-present
  Shape: (4140, 5)

  fundamental_raw.csv
  Quarterly financials, 2012-present
  Shape: (49, 29)

  macro_raw.csv
  Macro indicators, 2010-present
  Shape: (4351, 12)

  3/3 datasets collected

  Ready for Part 2: Feature Engineering
